# Проверка raw CRM-объектов для одного звонка

Минимальный notebook для Omega.

Задача: для одного `ucid` проверить, сколько **сырых CRM-строк** и сколько
**разных CRM-объектов** реально есть в `part-*.csv` по клиенту, связанному со звонком.

Это нужно для развилки:

- если raw даёт примерно столько же distinct task/offer объектов, сколько внешний пакет,
  недобора истории нет;
- если raw даёт заметно больше distinct объектов, пакет где-то потерял часть истории;
- если по одному `productOfferId` / `taskid` есть несколько строк с разными `version`
  или `CTL_VALIDFROM`, raw хранит версии/снапшоты;
- если по объекту одна строка, у нас только текущее состояние объекта плюс даты
  `creationTime` / `offerStartDate` / `updateTime`.

Notebook ничего не отправляет в API и не использует LLM.

In [ ]:
from __future__ import annotations

from pathlib import Path
import ast
import csv
import re
import warnings
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 280)

## 1. Настройки

In [ ]:
BASE_DIR = Path.cwd()

# Можно заменить на полный ucid или уникальный фрагмент.
TARGET_UCID_CONTAINS = "55af3ede"

# Если нужно искать не по ucid, можно заполнить один из ключей.
TARGET_ID_TASK = ""
TARGET_ID_PRPR = ""

# Если автоопределение ошибётся, задайте файлы явно.
VOICE_MATCH_XLSX = None  # например BASE_DIR / "Для метчинга.xlsx"
CRM_CSV_FILES = None     # например [BASE_DIR / "part-....csv", BASE_DIR / "part-....csv"]

# Окно, которое мы использовали для анализа декабрь-февраль.
CRM_WINDOW_START = pd.Timestamp("2025-12-01")
CRM_WINDOW_END_EXCLUSIVE = pd.Timestamp("2026-03-01")

OUT_DIR = BASE_DIR / "omega_one_call_raw_object_probe"
OUT_DIR.mkdir(exist_ok=True)

## 2. Вспомогательные функции

In [ ]:
def canon_name(value: Any) -> str:
    s = str(value).strip().lower()
    s = s.replace("ё", "е")
    s = re.sub(r"[\s_\-()/\\.,;:]+", "", s)
    return s


def clean(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    s = str(value).strip()
    if s.upper() in {"NULL", "[NULL]", "NONE", "NAN", "NA"}:
        return ""
    return s.strip().strip("'").strip('"')


def extract_ids(value: Any) -> list[str]:
    s = clean(value)
    if not s:
        return []
    vals: list[str] = []
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set)):
                vals = [clean(x) for x in parsed]
            else:
                vals = [clean(parsed)]
        except Exception:
            vals = re.findall(r"[A-Za-z0-9][A-Za-z0-9_\-]{3,}", s)
    else:
        vals = re.findall(r"[A-Za-z0-9][A-Za-z0-9_\-]{3,}", s)
        if not vals:
            vals = [s]
    out: list[str] = []
    seen: set[str] = set()
    for item in vals:
        item = clean(item)
        if item and item not in seen:
            out.append(item)
            seen.add(item)
    return out


def ids_key(values: list[str]) -> str:
    return " | ".join(values)


def find_col(df: pd.DataFrame, *, exact: list[str] | None = None, regex: list[str] | None = None, required: bool = False, label: str = "") -> str | None:
    exact = exact or []
    regex = regex or []
    exact_canons = {canon_name(x) for x in exact}
    for col in df.columns:
        if canon_name(col) in exact_canons:
            return col
    for pattern in regex:
        rx = re.compile(pattern, flags=re.I)
        for col in df.columns:
            if rx.search(str(col)):
                return col
    if required:
        raise KeyError(f"Не найдена обязательная колонка {label or exact or regex}. Доступные: {list(df.columns)}")
    return None


def parse_dt(value: Any) -> pd.Timestamp:
    if value is None or pd.isna(value):
        return pd.NaT
    s = clean(value)
    if not s:
        return pd.NaT
    return pd.to_datetime(s, errors="coerce")


def first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    by_canon = {canon_name(c): c for c in df.columns}
    for candidate in candidates:
        col = by_canon.get(canon_name(candidate))
        if col:
            return col
    return None


def value_counts_nonempty(series: pd.Series, n: int = 20) -> pd.DataFrame:
    s = series.map(clean)
    vc = s[s.ne("")].value_counts().head(n).reset_index()
    vc.columns = ["value", "n"]
    return vc


def write_xlsx(path: Path, sheets: dict[str, pd.DataFrame]) -> None:
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, df in sheets.items():
            safe_name = name[:31]
            df.to_excel(writer, sheet_name=safe_name, index=False)
            ws = writer.book[safe_name]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for col_cells in ws.columns:
                max_len = 10
                for cell in list(col_cells)[:200]:
                    max_len = max(max_len, min(80, len(str(cell.value or "")) + 2))
                ws.column_dimensions[col_cells[0].column_letter].width = max_len
    print("saved:", path)

## 3. Читаем Voice-ключи и CRM raw

In [ ]:
def autodetect_voice_file() -> Path:
    if VOICE_MATCH_XLSX:
        return Path(VOICE_MATCH_XLSX)
    candidates = sorted(
        list(BASE_DIR.glob("*метчинг*.xlsx"))
        + list(BASE_DIR.glob("*match*.xlsx"))
        + list(BASE_DIR.glob("*ключ*.xlsx"))
        + list(BASE_DIR.glob("*Для метчинга*.xlsx"))
    )
    if not candidates:
        raise FileNotFoundError("Не нашёл Excel с ключами Voice. Задайте VOICE_MATCH_XLSX явно.")
    # Предпочитаем явно подготовленный файл.
    candidates = sorted(candidates, key=lambda p: ("для метчинга" not in p.name.lower(), len(p.name)))
    return candidates[0]


def autodetect_crm_files() -> list[Path]:
    if CRM_CSV_FILES:
        return [Path(x) for x in CRM_CSV_FILES]
    files = sorted(BASE_DIR.glob("part-*.csv"))
    if not files:
        raise FileNotFoundError("Не нашёл part-*.csv. Задайте CRM_CSV_FILES явно.")
    return files


voice_path = autodetect_voice_file()
crm_paths = autodetect_crm_files()

voice = pd.read_excel(voice_path, dtype=str)
print("Voice match file:", voice_path.name, voice.shape)
print("CRM files:", [p.name for p in crm_paths])
display(voice.head(5))

In [ ]:
def read_crm_csv(path: Path) -> tuple[pd.DataFrame, dict[str, Any]]:
    read_log: dict[str, Any] = {"file": path.name}
    raw_lines = 0
    with path.open("r", encoding="cp1251", errors="replace") as f:
        raw_lines = sum(1 for _ in f)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", pd.errors.ParserWarning)
        df = pd.read_csv(
            path,
            sep="\t",
            encoding="cp1251",
            engine="python",
            quoting=csv.QUOTE_NONE,
            on_bad_lines="warn",
            dtype=str,
        )
    skipped = 0
    messages = []
    for warn in caught:
        msg = str(warn.message)
        messages.append(msg)
        m = re.search(r"Skipping line (\\d+):", msg)
        if m:
            skipped += 1
    read_log.update({
        "raw_lines_including_header": raw_lines,
        "rows_read": len(df),
        "columns": df.shape[1],
        "estimated_skipped_lines": max(raw_lines - 1 - len(df), skipped, 0),
        "parser_warnings": " || ".join(messages[:10]),
    })
    return df, read_log


crm_frames: dict[str, pd.DataFrame] = {}
read_logs = []
for path in crm_paths:
    df, log = read_crm_csv(path)
    crm_frames[path.name] = df
    read_logs.append(log)
    print(path.name, df.shape)

crm_read_log = pd.DataFrame(read_logs)
display(crm_read_log)

## 4. Определяем колонки и тип CRM-файлов

In [ ]:
ucid_col = find_col(voice, exact=["ucid"], regex=[r"ucid"], required=True, label="voice ucid")
call_date_col = find_col(voice, exact=["Дата звонка", "Дата активности"], regex=[r"дат.*(звон|актив)"], required=False, label="voice date")
id_task_col = find_col(voice, exact=["Id задачи", "ID задачи", "id task", "taskid"], regex=[r"(id|ed).*(задач|task)|taskid"], required=False, label="voice task id")
id_prpr_col = find_col(voice, exact=["Id ПрПр", "ID ПрПр", "productOfferId"], regex=[r"(id|ed).*(прпр|offer)|productoffer"], required=False, label="voice offer id")
org_col = find_col(voice, exact=["Id Организации (стп/ЕКП)", "Id Организации"], regex=[r"(id|ed).*(орган|org|екп|стп)"], required=False, label="voice org id")
call_id_col = find_col(voice, exact=["Id звонка", "ID звонка"], regex=[r"(id|ed).*(звон|call)"], required=False, label="voice call id")

print("Voice columns detected:")
for label, col in {
    "ucid": ucid_col,
    "call_date": call_date_col,
    "id_task": id_task_col,
    "id_prpr": id_prpr_col,
    "org": org_col,
    "call_id": call_id_col,
}.items():
    print(f"  {label}: {col}")

In [ ]:
def profile_crm_frame(name: str, df: pd.DataFrame) -> dict[str, Any]:
    return {
        "name": name,
        "rows": len(df),
        "columns": df.shape[1],
        "taskid_col": first_existing(df, ["taskid", "taskId"]),
        "productOfferId_col": first_existing(df, ["productOfferId", "productofferid"]),
        "key_col": first_existing(df, ["key"]),
        "ucpid_col": first_existing(df, ["ucpid", "ucpId", "ucpID"]),
        "creation_col": first_existing(df, ["creationTime", "createdTime", "createdAt"]),
        "update_col": first_existing(df, ["updateTime", "updatedTime", "updatedAt", "lastUpdateTime"]),
        "offer_start_col": first_existing(df, ["offerStartDate", "startDate"]),
        "version_col": first_existing(df, ["version"]),
        "valid_from_col": first_existing(df, ["CTL_VALIDFROM", "validFrom"]),
        "valid_to_col": first_existing(df, ["CTL_VALIDTO", "validTo"]),
        "product_group_col": first_existing(df, ["productGroupCode", "product_group_code", "product"]),
        "product_code_col": first_existing(df, ["productCode", "product_code"]),
        "status_col": first_existing(df, ["statusCode", "status_code"]),
        "stage_col": first_existing(df, ["stageCode", "stage_code"]),
        "result_col": first_existing(df, ["resultCode", "result_code"]),
        "refusal_col": first_existing(df, ["refusalCode", "refusal_code"]),
    }


profiles = [profile_crm_frame(name, df) for name, df in crm_frames.items()]
crm_profiles = pd.DataFrame(profiles)
display(crm_profiles)

task_profile = next((p for p in profiles if p["taskid_col"]), None)
offer_profile = next((p for p in profiles if p["productOfferId_col"]), None)
if task_profile is None or offer_profile is None:
    raise RuntimeError("Не смог определить task/offer CRM-файлы. Проверьте CRM profiles.")

task_df = crm_frames[task_profile["name"]].copy()
offer_df = crm_frames[offer_profile["name"]].copy()
print("Task file:", task_profile["name"])
print("Offer file:", offer_profile["name"])

## 5. Находим целевой звонок и CRM-клиента

In [ ]:
def row_ids(row: pd.Series, col: str | None) -> list[str]:
    if not col:
        return []
    return extract_ids(row.get(col, ""))


mask = pd.Series(False, index=voice.index)
if TARGET_UCID_CONTAINS:
    mask |= voice[ucid_col].map(clean).str.contains(re.escape(TARGET_UCID_CONTAINS), case=False, na=False)
if TARGET_ID_TASK and id_task_col:
    mask |= voice[id_task_col].apply(lambda x: TARGET_ID_TASK in extract_ids(x))
if TARGET_ID_PRPR and id_prpr_col:
    mask |= voice[id_prpr_col].apply(lambda x: TARGET_ID_PRPR in extract_ids(x))

target_rows = voice[mask].copy()
if target_rows.empty:
    raise ValueError("Целевой звонок не найден. Проверьте TARGET_UCID_CONTAINS / TARGET_ID_TASK / TARGET_ID_PRPR.")

target = target_rows.iloc[0].copy()
target_ucid = clean(target[ucid_col])
target_call_date = parse_dt(target[call_date_col]) if call_date_col else pd.NaT
target_task_ids = row_ids(target, id_task_col)
target_offer_ids = row_ids(target, id_prpr_col)
target_org_ids = row_ids(target, org_col)

print("Target rows:", len(target_rows))
print("target ucid:", target_ucid)
print("target call date:", target_call_date)
print("target task ids:", target_task_ids)
print("target offer ids:", target_offer_ids)
print("target org ids:", target_org_ids[:10])
display(target_rows)

In [ ]:
def find_rows_by_any_id(df: pd.DataFrame, cols: list[str | None], ids: list[str]) -> pd.DataFrame:
    ids_clean = {clean(x) for x in ids if clean(x)}
    if not ids_clean:
        return df.iloc[0:0].copy()
    mask = pd.Series(False, index=df.index)
    for col in cols:
        if not col or col not in df.columns:
            continue
        mask |= df[col].apply(lambda x: bool(ids_clean.intersection(extract_ids(x))))
    return df[mask].copy()


task_direct = find_rows_by_any_id(task_df, [task_profile["taskid_col"], task_profile["key_col"]], target_task_ids)
offer_direct = find_rows_by_any_id(offer_df, [offer_profile["productOfferId_col"], offer_profile["key_col"]], target_offer_ids)

candidate_ucpids: list[str] = []
for frame, profile in [(task_direct, task_profile), (offer_direct, offer_profile)]:
    ucpid_col = profile.get("ucpid_col")
    if ucpid_col and ucpid_col in frame.columns:
        for value in frame[ucpid_col].tolist():
            candidate_ucpids.extend(extract_ids(value))

# Если в Voice есть организация, тоже пробуем как client key, но это fallback.
candidate_ucpids.extend(target_org_ids)
candidate_ucpids = list(dict.fromkeys([x for x in candidate_ucpids if clean(x)]))

bridge_summary = pd.DataFrame([
    {"metric": "target_ucid", "value": target_ucid},
    {"metric": "target_call_date", "value": str(target_call_date)},
    {"metric": "voice_task_ids", "value": ids_key(target_task_ids)},
    {"metric": "voice_offer_ids", "value": ids_key(target_offer_ids)},
    {"metric": "voice_org_ids_first10", "value": ids_key(target_org_ids[:10])},
    {"metric": "direct_task_rows", "value": len(task_direct)},
    {"metric": "direct_offer_rows", "value": len(offer_direct)},
    {"metric": "candidate_ucpids", "value": ids_key(candidate_ucpids)},
])

display(bridge_summary)
display(task_direct.head(20))
display(offer_direct.head(20))

## 6. Вытаскиваем всю raw-историю CRM по найденному клиенту

In [ ]:
def rows_by_ucpid(df: pd.DataFrame, profile: dict[str, Any], ucpids: list[str]) -> pd.DataFrame:
    ucpid_col = profile.get("ucpid_col")
    if not ucpid_col or ucpid_col not in df.columns or not ucpids:
        return df.iloc[0:0].copy()
    ucpids_clean = {clean(x) for x in ucpids if clean(x)}
    mask = df[ucpid_col].apply(lambda x: bool(ucpids_clean.intersection(extract_ids(x))))
    return df[mask].copy()


task_client = rows_by_ucpid(task_df, task_profile, candidate_ucpids)
offer_client = rows_by_ucpid(offer_df, offer_profile, candidate_ucpids)
task_client["crm_entity"] = "task"
offer_client["crm_entity"] = "offer"
task_client["crm_row_index"] = task_client.index
offer_client["crm_row_index"] = offer_client.index

print("task rows for client:", len(task_client))
print("offer rows for client:", len(offer_client))
display(task_client.head(10))
display(offer_client.head(10))

In [ ]:
def object_id_series(df: pd.DataFrame, profile: dict[str, Any], entity: str) -> pd.Series:
    primary = profile.get("taskid_col") if entity == "task" else profile.get("productOfferId_col")
    fallback = profile.get("key_col")
    out = pd.Series("", index=df.index, dtype=object)
    if primary and primary in df.columns:
        out = df[primary].apply(lambda x: extract_ids(x)[0] if extract_ids(x) else "")
    if fallback and fallback in df.columns:
        missing = out.map(clean).eq("")
        out.loc[missing] = df.loc[missing, fallback].apply(lambda x: extract_ids(x)[0] if extract_ids(x) else "")
    missing = out.map(clean).eq("")
    out.loc[missing] = [f"{entity}_row_{idx}" for idx in df.index[missing]]
    return out


def add_dates(df: pd.DataFrame, profile: dict[str, Any]) -> pd.DataFrame:
    df = df.copy()
    date_cols = []
    for key in ["creation_col", "offer_start_col", "update_col", "valid_from_col", "valid_to_col"]:
        col = profile.get(key)
        if col and col in df.columns:
            df[f"__dt_{col}"] = df[col].apply(parse_dt)
            date_cols.append(f"__dt_{col}")
    if date_cols:
        df["__min_known_date"] = df[date_cols].min(axis=1)
        df["__max_known_date"] = df[date_cols].max(axis=1)
    else:
        df["__min_known_date"] = pd.NaT
        df["__max_known_date"] = pd.NaT
    return df


task_client = add_dates(task_client, task_profile)
offer_client = add_dates(offer_client, offer_profile)
task_client["__object_id"] = object_id_series(task_client, task_profile, "task")
offer_client["__object_id"] = object_id_series(offer_client, offer_profile, "offer")


def in_analysis_window(df: pd.DataFrame) -> pd.Series:
    # Для диагностики используем мягкое окно: объект относится к окну, если любая известная дата попала в декабрь-февраль.
    date_cols = [c for c in df.columns if c.startswith("__dt_")]
    if not date_cols:
        return pd.Series(False, index=df.index)
    mask = pd.Series(False, index=df.index)
    for col in date_cols:
        mask |= df[col].ge(CRM_WINDOW_START) & df[col].lt(CRM_WINDOW_END_EXCLUSIVE)
    return mask.fillna(False)


task_client["__in_analysis_window"] = in_analysis_window(task_client)
offer_client["__in_analysis_window"] = in_analysis_window(offer_client)

## 7. Считаем строки, distinct-объекты и версии/дубли

In [ ]:
def summarize_objects(df: pd.DataFrame, entity: str) -> dict[str, Any]:
    in_win = df[df["__in_analysis_window"]].copy()
    counts = df.groupby("__object_id", dropna=False).size().reset_index(name="raw_rows_per_object")
    counts_win = in_win.groupby("__object_id", dropna=False).size().reset_index(name="raw_rows_per_object_in_window")
    multi = counts[counts["raw_rows_per_object"].gt(1)]
    return {
        "entity": entity,
        "raw_rows_all": len(df),
        "raw_rows_in_window": len(in_win),
        "distinct_objects_all": df["__object_id"].nunique(dropna=True) if len(df) else 0,
        "distinct_objects_in_window": in_win["__object_id"].nunique(dropna=True) if len(in_win) else 0,
        "objects_with_multiple_raw_rows": len(multi),
        "max_raw_rows_per_object": int(counts["raw_rows_per_object"].max()) if len(counts) else 0,
        "objects_with_multiple_raw_rows_in_window": int(counts_win["raw_rows_per_object_in_window"].gt(1).sum()) if len(counts_win) else 0,
    }


summary = pd.DataFrame([
    summarize_objects(task_client, "task"),
    summarize_objects(offer_client, "offer"),
])
summary.loc[len(summary)] = {
    "entity": "total",
    "raw_rows_all": int(summary["raw_rows_all"].sum()),
    "raw_rows_in_window": int(summary["raw_rows_in_window"].sum()),
    "distinct_objects_all": int(summary["distinct_objects_all"].sum()),
    "distinct_objects_in_window": int(summary["distinct_objects_in_window"].sum()),
    "objects_with_multiple_raw_rows": int(summary["objects_with_multiple_raw_rows"].sum()),
    "max_raw_rows_per_object": int(summary["max_raw_rows_per_object"].max()),
    "objects_with_multiple_raw_rows_in_window": int(summary["objects_with_multiple_raw_rows_in_window"].sum()),
}
display(summary)

In [ ]:
def duplicate_version_view(df: pd.DataFrame, profile: dict[str, Any], entity: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    counts = df.groupby("__object_id", dropna=False).size().reset_index(name="raw_rows_per_object")
    multi_ids = set(counts[counts["raw_rows_per_object"].gt(1)]["__object_id"])
    cols = ["__object_id", "crm_entity", "crm_row_index", "__in_analysis_window"]
    for key in ["taskid_col", "productOfferId_col", "key_col", "ucpid_col", "creation_col", "offer_start_col", "update_col", "version_col", "valid_from_col", "valid_to_col", "product_group_col", "status_col", "stage_col", "result_col"]:
        col = profile.get(key)
        if col and col in df.columns and col not in cols:
            cols.append(col)
    if not multi_ids:
        return pd.DataFrame(columns=cols + ["note"])
    out = df[df["__object_id"].isin(multi_ids)][cols].copy()
    return out.sort_values(["__object_id", "crm_row_index"])


task_versions = duplicate_version_view(task_client, task_profile, "task")
offer_versions = duplicate_version_view(offer_client, offer_profile, "offer")
print("Task duplicate/version rows:", len(task_versions))
print("Offer duplicate/version rows:", len(offer_versions))
display(task_versions.head(50))
display(offer_versions.head(50))

## 8. Проверяем relaxed SALARY-карточку до звонка

In [ ]:
def contains_salary_text(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    cols = [c for c in cols if c and c in df.columns]
    if not cols:
        return pd.Series(False, index=df.index)
    text = df[cols].fillna("").astype(str).agg(" ".join, axis=1)
    return text.str.contains(r"SALARY|ЗАРПЛАТ|зарплат|реестр|сотрудник|ведомост|карты сотруд", case=False, na=False)


salary_probe = offer_client.copy()
offer_product_cols = [
    offer_profile.get("product_group_col"),
    offer_profile.get("product_code_col"),
    offer_profile.get("status_col"),
    offer_profile.get("stage_col"),
    offer_profile.get("result_col"),
    "productOfferDescription",
    "closingComment",
    "offerDeactivationComment",
]
salary_probe["__is_salary"] = contains_salary_text(salary_probe, [c for c in offer_product_cols if c])

creation_col = offer_profile.get("creation_col")
offer_start_col = offer_profile.get("offer_start_col")
update_col = offer_profile.get("update_col")

salary_probe["__created_before_call"] = False
salary_probe["__offer_started_before_call"] = False
salary_probe["__updated_after_call"] = False

if pd.notna(target_call_date):
    if creation_col and f"__dt_{creation_col}" in salary_probe.columns:
        salary_probe["__created_before_call"] = salary_probe[f"__dt_{creation_col}"].lt(target_call_date)
    if offer_start_col and f"__dt_{offer_start_col}" in salary_probe.columns:
        salary_probe["__offer_started_before_call"] = salary_probe[f"__dt_{offer_start_col}"].lt(target_call_date)
    if update_col and f"__dt_{update_col}" in salary_probe.columns:
        salary_probe["__updated_after_call"] = salary_probe[f"__dt_{update_col}"].ge(target_call_date)

salary_probe["__salary_card_before_call_relaxed"] = (
    salary_probe["__is_salary"]
    & (salary_probe["__created_before_call"] | salary_probe["__offer_started_before_call"])
)

keep_cols = [
    "__object_id",
    offer_profile.get("productOfferId_col"),
    offer_profile.get("key_col"),
    offer_profile.get("ucpid_col"),
    offer_profile.get("product_group_col"),
    offer_profile.get("product_code_col"),
    creation_col,
    offer_start_col,
    update_col,
    offer_profile.get("version_col"),
    offer_profile.get("valid_from_col"),
    offer_profile.get("valid_to_col"),
    offer_profile.get("status_col"),
    offer_profile.get("stage_col"),
    offer_profile.get("result_col"),
    "__is_salary",
    "__created_before_call",
    "__offer_started_before_call",
    "__updated_after_call",
    "__salary_card_before_call_relaxed",
]
keep_cols = [c for c in keep_cols if c and c in salary_probe.columns]
salary_probe_view = salary_probe[keep_cols].copy()

print("Relaxed SALARY cards before call:", int(salary_probe["__salary_card_before_call_relaxed"].sum()))
display(salary_probe_view)

## 9. Сохраняем результат

In [ ]:
def strip_internal_dt(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[c for c in df.columns if c.startswith("__dt_")], errors="ignore")


write_xlsx(
    OUT_DIR / "one_call_raw_object_probe.xlsx",
    {
        "Target voice row": target_rows,
        "Bridge summary": bridge_summary,
        "CRM read log": crm_read_log,
        "CRM profiles": crm_profiles,
        "Distinct summary": summary,
        "Task raw for client": strip_internal_dt(task_client),
        "Offer raw for client": strip_internal_dt(offer_client),
        "Task duplicate groups": task_versions,
        "Offer duplicate groups": offer_versions,
        "Relaxed salary probe": salary_probe_view,
    },
)

print("Готово. Главный файл:", OUT_DIR / "one_call_raw_object_probe.xlsx")

## Как читать результат

1. `Bridge summary` — через какой ключ целевой звонок вышел на CRM-клиента:
   `Id задачи`, `Id ПрПр` или `Id организации`.
2. `Distinct summary` — главная таблица:
   `raw_rows_all` = сколько строк raw нашлось по клиенту;
   `distinct_objects_all` = сколько разных task/offer объектов;
   `objects_with_multiple_raw_rows` = есть ли дубли/версии одного объекта.
3. `Task duplicate groups` / `Offer duplicate groups`:
   если пусто, raw не хранит несколько строк на один объект;
   если не пусто, смотрим `version`, `CTL_VALIDFROM`, даты.
4. `Relaxed salary probe` — показывает, была ли карточка SALARY/ЗП создана или начата до звонка.
   Если `__salary_card_before_call_relaxed = True`, но `updateTime` после звонка,
   это можно использовать только как existence-only факт: карточка была, но её текущее
   содержимое нельзя считать предзвонковым состоянием.